In [1]:
!pip install -q spacy transformers nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import json
import pandas as pd
import spacy
import nltk
import re

from transformers import pipeline
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
nlp = spacy.load("en_core_web_sm")

sentiment_model = pipeline("sentiment-analysis")

topic_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
file_path = "/content/drive/MyDrive/aiassignment3/CrimeReport (1).txt"

records = []

with open(file_path, "r") as f:
    for line in f:
        try:
            data = json.loads(line.strip())

            text = data.get("text", "")
            location = data.get("user", {}).get("location", "")

            records.append({
                "text": text,
                "location": location
            })

        except:
            continue

df = pd.DataFrame(records)
df.head()

,text,location
0,Active crime scene on I-59/20 near Jeff/Tusc C...,Alabama
1,Police have arrested a suspect in the Monday s...,"Rocky Mount, N.C."
2,Lawsuit alleges Chicago Police strip-searched ...,Northern Virginia
3,New NRA Advice: Don't Cooperate With Police If...,
4,"Police failing to stamp out ‘honour crimes’, f...",London


In [7]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [8]:
def extract_location(text, fallback_location):
    doc = nlp(text)

    locations = [
        ent.text for ent in doc.ents
        if ent.label_ in ["GPE", "LOC"]
    ]

    if locations:
        return locations[0]

    return fallback_location if fallback_location else "Unknown"

In [9]:
labels = [
    "robbery",
    "theft",
    "assault",
    "murder",
    "accident",
    "fire",
    "fraud",
    "disturbance"
]

def classify_topic(text):
    if len(text.strip()) < 10:
        return "Unknown"

    result = topic_model(text[:512], labels)
    return result["labels"][0]

In [10]:
def get_sentiment(text):
    result = sentiment_model(text[:512])[0]

    if "NEG" in result["label"]:
        return "Negative"
    elif "POS" in result["label"]:
        return "Positive"
    return "Neutral"

In [11]:
def get_severity(topic):
    topic = topic.lower()

    if topic in ["murder", "assault", "robbery"]:
        return "High"
    elif topic in ["theft", "accident", "disturbance"]:
        return "Medium"
    return "Low"

In [12]:
results = []

for i, row in df.iterrows():
    text = row["text"]
    location_raw = row["location"]

    if not text or len(text.strip()) < 5:
        continue

    topic = classify_topic(text)
    sentiment = get_sentiment(text)
    location = extract_location(text, location_raw)
    severity = get_severity(topic)

    results.append({
        "Text_ID": f"TXT_{i+1:03}",
        "Crime_Type": topic.title(),
        "Location_Entity": location,
        "Sentiment": sentiment,
        "Topic": topic.title(),
        "Severity_Label": severity
    })

final_df = pd.DataFrame(results)
final_df.head(20)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,Text_ID,Crime_Type,Location_Entity,Sentiment,Topic,Severity_Label
0,TXT_001,Disturbance,Alabama,Negative,Disturbance,Medium
1,TXT_002,Disturbance,"Rocky Mount, N.C.",Negative,Disturbance,Medium
2,TXT_003,Disturbance,Chicago,Negative,Disturbance,Medium
3,TXT_004,Disturbance,Unknown,Negative,Disturbance,Medium
4,TXT_005,Murder,London,Negative,Murder,High
5,TXT_006,Disturbance,London,Negative,Disturbance,Medium
6,TXT_007,Robbery,"Buffalo, NY",Negative,Robbery,High
7,TXT_008,Disturbance,Pakistan,Negative,Disturbance,Medium
8,TXT_009,Disturbance,"Washington, DC",Negative,Disturbance,Medium
9,TXT_010,Disturbance,Baltimore,Positive,Disturbance,Medium


In [13]:
output_path = "/content/drive/MyDrive/aiassignment3/student5_output.csv"
final_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/aiassignment3/student5_output.csv
